### Versteigerung – Lösung Schritt für Schritt

*32. Bundeswettbewerb Informatik (2013), 1. Runde, Aufgabe 5*

Die Herstellung des Materials Sehrteurium ist aufwendig, und das Verfahren ist streng geheim. Die beiden Abnehmer Alphasoft und Betahard benötigen das Material dringend für ihre Produktion. Der Hersteller von Sehrteurium versteigert am Ende jeden Monats die monatliche Produktionsmenge von 20 Gramm nach folgenden Regeln:

Die Vertreter von Alphasoft und Betahard müssen zur Versteigerung mit je 1000 Unzen Gold erscheinen. Sie bieten dann gleichzeitig eine frei wählbare Anzahl ihrer Goldunzen auf die ersten 2 Gramm Sehrteurium. Nach der Abgabe werden die Gebote für beide sichtbar. Den Zuschlag erhält, wer die meisten Unzen geboten hat; bei Gleichstand erhält jeder Bieter 1 Gramm. Beide Bieter müssen ihren jeweiligen gebotenen Betrag bezahlen – auch der unterlegene. Ein Gebot von 0 Unzen ist erlaubt. Das Bieten auf jeweils zwei Gramm wird solange wiederholt, bis der Vorrat an Sehrteurium komplett versteigert ist.

Jeder Bieter möchte selbstverständlich eine größere Menge ersteigern als sein Konkurrent. Deshalb hat die Firma Alphasoft ein Computerprogramm entwickelt, das für jede Situation ein Gebot berechnet. Firma Betahard möchte ebenfalls ein solches Programm besitzen und schreibt ein Turnier aus, in dem Programme in simulierten Versteigerungen gegeneinander antreten. In einer Versteigerung gewinnt das Programm, das in der Lage ist, mehr Sehrteurium zu ersteigern als das andere. Bei Gleichstand gewinnt das Programm, das mehr Gold übrig behält.

#### Aufgabe 5

Schreibe ein Programm, welches an solch einer Versteigerung teilnehmen kann, und erläutere seine Strategie.

Miss dein Programm im BWINF-Turniersystem mit anderen Programmen; auch das Alphasoft-Programm ist dort zu finden. Du hast die Aufgabe erfolgreich gelöst, wenn dein Programm das Alphasoft-Programm meistens schlägt. Dieses bietet immer den gleichen Betrag: 100 Unzen.

Nach dem Einsendeschluss wird ein Turnier durchgeführt, in dem jedes Programm gegen jedes andere zweimal antritt. Die Bestplatzierten des Turniers erhalten einen Sonderpreis.

**Zu den Daten:** Das Turniersystem gibt es nicht mehr, und Beispieldaten gibt es nicht. Wir bauen das Spiel selbst nach und programmieren die Gegner, die in den Lösungshinweisen beschrieben sind, selbst.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Spiel nachbauen (Simulation):** Zehn Runden, in jeder bieten beide Spieler, dann werden Gramm und Gold verrechnet.
- **Einfache Überlegung statt Rechnen:** Es gibt 10 Runden mit je 2 Gramm. Wer sechs Runden gewinnt, hat 12 von 20 Gramm und damit sicher gewonnen. Die ganze Aufgabe lässt sich auf diese Zahl 6 zurückführen.
- **Aus der Vergangenheit lernen (adaptive Strategie):** Man sieht die bisherigen Gebote des Gegners und schätzt daraus sein nächstes Gebot.
- **Turnier „jeder gegen jeden“:** Ein Programm ist nur so gut wie seine Gegner. Wir lassen alle gegeneinander antreten und werten aus.
- **Zufallsgegner:** Gegen sie sagt ein einzelnes Spiel wenig; man spielt viele Spiele mit festem Startwert für den Zufall und zählt.

**Nützliche Funktionen und Bibliotheken**
- `import random`: `random.randint(a, b)` (Zufallszahl von a bis b einschließlich), `random.seed(n)` (jedes Mal dieselben Zufallszahlen).
- `min(a, b)`, `max(a, b)`: man darf nie mehr bieten, als man noch hat.
- `zip(liste1, liste2)`: zwei Listen gleichzeitig durchlaufen (die Gebote der beiden Spieler Runde für Runde).
- `sum(liste)`: Summe, z. B. gebotenes Gold.
- Strategien werden hier über ihren **Namen** (Text) ausgewählt; eine Funktion `gebot` verteilt dann an die richtige Strategie.

## Die Idee in Worten

**Das Spiel.** Es gibt 20 Gramm, also 10 Runden mit je 2 Gramm. Jeder hat 1000 Unzen. Wer in einer Runde mehr bietet, bekommt die 2 Gramm; beide zahlen ihr Gebot. Gewonnen hat, wer am Ende mehr Gramm hat, bei gleich viel Gramm, wer mehr Gold übrig hat.

**Alphasoft** bietet in jeder der 10 Runden 100 Unzen, zusammen genau 1000. Wer mehr als 100 bietet, gewinnt diese Runde. Gewinnt man 6 Runden, hat man mindestens 12 Gramm und Alphasoft höchstens 8. Dafür reicht zum Beispiel 6-mal 101 Unzen, zusammen 606 Unzen; in den übrigen 4 Runden bietet man nichts oder wenig. So schlägt man Alphasoft **immer**, nicht nur meistens.

**Aber:** Gegen andere Gegner hilft das nicht. Wer immer 101 bietet, verliert gegen einen Gegner, der in sechs Runden 166 bietet. Die Lösungshinweise unterscheiden daher Klassen von Gegnern (konstant, linear steigend oder fallend, „immer eins mehr“, zufällig) und schlagen zwei Arten von Programmen vor:
1. eine **feste Gebotsfolge**, die möglichst viele dieser Gegner schlägt („Die Alternative“): hohe und niedrige Gebote im Wechsel, nie 0, mindestens 6-mal über 100, am Ende noch Gold übrig;
2. ein **adaptives Programm**, das aus den ersten Geboten des Gegners auf die weiteren schließt.

Wir bauen beides, lassen alle gegeneinander antreten und sehen, wer wen schlägt. Zum Schluss lassen wir MiniZinc eine feste Folge suchen, die möglichst viele vorgegebene Gegnerfolgen schlägt.

## Das Spiel nachbauen

Ein Spiel besteht aus zwei Listen mit je 10 Geboten. Die Funktion `gramm` zählt für einen Spieler die Gramm: 2 für jede Runde, in der er mehr bietet, 1 bei Gleichstand. Die Funktion `ergebnis` sagt, wer gewinnt: `1` (der erste Spieler), `-1` (der zweite) oder `0` (Unentschieden: gleich viele Gramm und gleich viel Gold übrig).

In [ ]:
import random


def gramm(meine, ihre):
    """Gramm Sehrteurium des einen Spielers, wenn er meine und der Gegner ihre Gebote bietet."""
    summe = 0
    for a, b in zip(meine, ihre):
        if a > b:
            summe = summe + 2
        elif a == b:
            summe = summe + 1
    return summe


def ergebnis(gebote_a, gebote_b):
    """1: A gewinnt, -1: B gewinnt, 0: unentschieden."""
    ga = gramm(gebote_a, gebote_b)
    gb = gramm(gebote_b, gebote_a)
    gold_a = 1000 - sum(gebote_a)
    gold_b = 1000 - sum(gebote_b)
    if ga > gb or (ga == gb and gold_a > gold_b):
        return 1
    if gb > ga or (ga == gb and gold_b > gold_a):
        return -1
    return 0


alphasoft = [100] * 10
sechsmal = [101, 101, 101, 101, 101, 101, 0, 0, 0, 0]
print(gramm(sechsmal, alphasoft), 'zu', gramm(alphasoft, sechsmal), 'Gramm, Gold übrig:', 1000 - sum(sechsmal))
print('Ergebnis (1 = wir gewinnen):', ergebnis(sechsmal, alphasoft))

12 zu 8 Gramm, Gold übrig: 394
Ergebnis (1 = wir gewinnen): 1


### Warum jede Folge mit mindestens 6 Geboten über 100 Alphasoft schlägt

Mindestens 6 Gebote über 100 heißt: mindestens 6 gewonnene Runden, also mindestens 12 Gramm. Alphasoft bekommt höchstens 4 · 2 = 8 Gramm. Zur Probe 5000 zufällige Folgen, die höchstens 1000 Unzen kosten und mindestens 6-mal über 100 liegen:

In [ ]:
random.seed(5)
geprueft = 0
while geprueft < 5000:
    folge = [random.randint(0, 250) for _ in range(10)]
    if sum(folge) <= 1000 and sum(1 for g in folge if g > 100) >= 6:
        assert ergebnis(folge, alphasoft) == 1
        geprueft = geprueft + 1
print(geprueft, 'Folgen geprüft, alle schlagen Alphasoft')

5000 Folgen geprüft, alle schlagen Alphasoft


## Die Gegner

Jeder Spieler bekommt einen Namen. Die Funktion `gebot(name, runde, meine, ihre, rest)` liefert sein Gebot in der Runde `runde` (1 bis 10), wenn `meine` und `ihre` die bisherigen Gebote (erst die des Spielers, dann die des Gegners) sind und `rest` das Gold, das der Spieler noch hat. Nie mehr bieten, als man hat: das letzte Stück der Funktion ist `min(g, rest)`.

Die Gegner nach den Lösungshinweisen:

| Name | Gebote |
|---|---|
| `alphasoft` | immer 100 |
| `konstant101` | 101 in den Runden 1 bis 9, 91 in Runde 10 (die „trivialste“ Lösung der Hinweise) |
| `sechs166` | 6-mal 166, danach nur 1 |
| `linear_auf` | 7, 27, 47, …, 187 (steigt um 20) |
| `linear_ab` | 187, 167, …, 7 (fällt um 20) |
| `mehr1` | immer 1 mehr als das letzte Gebot des Gegners („Immer-eins-mehr“) |
| `mehr20` | immer 20 mehr als das letzte Gebot des Gegners |
| `zufall` | zufällig zwischen 100 und 199 |
| `alternative` | feste Folge 10, 165, 101, 2, 166, 4, 14, 170, 142, 165 aus den Hinweisen („Die Alternative“) |
| `kopie` | bietet das letzte Gebot des Gegners (in Runde 1: 100) |

Die beiden **adaptiven** Spieler (`letztes`, `trend`) sind unsere eigenen Vorschläge:

- `letztes`: Schätze, dass der Gegner wieder so viel bietet wie zuletzt (in Runde 1: 100), und biete 1 mehr. Das darf aber nicht zu teuer werden: Wer noch `n` Runden gewinnen muss, um 11 Gramm zu erreichen, hat `rest / n` Unzen für jede dieser Runden. Ist die Schätzung höher, opfert man die Runde und bietet nur 1. Hat man schon 11 Gramm, ist alles gewonnen und man bietet 0.
- `trend`: Wie `letztes`, aber wenn die Gebote des Gegners steigen, rechnet man mit demselben Anstieg noch einmal (letztes Gebot + letzter Anstieg); fallen sie, nimmt man das letzte Gebot.

In [ ]:
FOLGE_ALTERNATIVE = [10, 165, 101, 2, 166, 4, 14, 170, 142, 165]


def noch_noetig(meine, ihre):
    """Wie viele Runden man noch gewinnen muss, um mindestens 11 Gramm zu haben (0: schon sicher gewonnen)."""
    fehlend = 11 - gramm(meine, ihre)
    if fehlend <= 0:
        return 0
    return (fehlend + 1) // 2


def adaptiv(meine, ihre, rest, mit_trend):
    noetig = noch_noetig(meine, ihre)
    if noetig == 0:
        return 0
    if len(ihre) == 0:
        schaetzung = 100
    else:
        schaetzung = ihre[-1]
        if mit_trend and len(ihre) >= 2 and ihre[-1] > ihre[-2]:
            schaetzung = ihre[-1] + (ihre[-1] - ihre[-2])
    schaetzung = schaetzung + 1
    if schaetzung * noetig <= rest:
        return schaetzung
    return 1


def gebot(name, runde, meine, ihre, rest):
    """Gebot des Spielers name in der Runde runde (1 bis 10)."""
    if name == 'alphasoft':
        g = 100
    elif name == 'konstant101':
        g = 101 if runde < 10 else 91
    elif name == 'sechs166':
        g = 166 if runde <= 6 else 1
    elif name == 'linear_auf':
        g = 7 + 20 * (runde - 1)
    elif name == 'linear_ab':
        g = 187 - 20 * (runde - 1)
    elif name == 'mehr1':
        g = (ihre[-1] if ihre else 0) + 1
    elif name == 'mehr20':
        g = (ihre[-1] if ihre else 0) + 20
    elif name == 'zufall':
        g = random.randint(100, 199)
    elif name == 'alternative':
        g = FOLGE_ALTERNATIVE[runde - 1]
    elif name == 'kopie':
        g = ihre[-1] if ihre else 100
    elif name == 'letztes':
        g = adaptiv(meine, ihre, rest, False)
    elif name == 'trend':
        g = adaptiv(meine, ihre, rest, True)
    return max(0, min(g, rest))


def spiele(name_a, name_b):
    """Ein Spiel über 10 Runden. Gibt die Gebote beider Spieler zurück."""
    gebote_a = []
    gebote_b = []
    rest_a = 1000
    rest_b = 1000
    for runde in range(1, 11):
        a = gebot(name_a, runde, gebote_a, gebote_b, rest_a)
        b = gebot(name_b, runde, gebote_b, gebote_a, rest_b)
        gebote_a.append(a)
        gebote_b.append(b)
        rest_a = rest_a - a
        rest_b = rest_b - b
    return gebote_a, gebote_b


for gegner in ['alphasoft', 'mehr1', 'linear_ab']:
    a, b = spiele('letztes', gegner)
    print('letztes  ', a)
    print(gegner.ljust(9), b, '-> Ergebnis', ergebnis(a, b), '\n')

letztes   [101, 101, 101, 101, 101, 101, 0, 0, 0, 0]
alphasoft [100, 100, 100, 100, 100, 100, 100, 100, 100, 100] -> Ergebnis 1 

letztes   [101, 2, 103, 4, 105, 6, 107, 8, 109, 10]
mehr1     [1, 102, 3, 104, 5, 106, 7, 108, 9, 110] -> Ergebnis 0 

letztes   [101, 1, 1, 148, 128, 108, 88, 68, 48, 0]
linear_ab [187, 167, 147, 127, 107, 87, 67, 47, 27, 7] -> Ergebnis 1 



Gegen Alphasoft bietet `letztes` 6-mal 101 und danach 0: sicher gewonnen mit 12 : 8 Gramm. Gegen `mehr1` überbieten sich beide abwechselnd um 1: jeder gewinnt 5 Runden, und beide zahlen zusammen genau gleich viel (je 555 Unzen), das Spiel endet **unentschieden**. Gegen `linear_ab` verliert `letztes` die erste Runde (101 gegen 187) und bietet dann nur 1; sobald der Gegner stark gefallen ist, gewinnt es die Runden 4 bis 9 billig und hat 12 Gramm.

## Das Turnier

Jeder spielt gegen jeden. Deterministische Paarungen spielt man einmal. Sind `zufall` oder ein anderer Zufallsspieler beteiligt, spielen wir 200 Spiele (fester Startwert für den Zufall) und geben den Anteil der gewonnenen Spiele an. Eine Zeile gehört dem Spieler links, die Spalte dem Gegner oben, die Zahl ist der Anteil der Spiele in Prozent, die der Spieler links gewinnt (Unentschieden zählen nicht als Sieg, daher nicht immer 100 − x gegenüber).

In [ ]:
namen = ['alphasoft', 'konstant101', 'sechs166', 'linear_auf', 'linear_ab', 'mehr1', 'mehr20', 'zufall',
         'alternative', 'kopie', 'letztes', 'trend']
random.seed(2013)
siegprozent = {}
for links in namen:
    for rechts in namen:
        anzahl = 200 if 'zufall' in (links, rechts) else 1
        siege = 0
        for _ in range(anzahl):
            a, b = spiele(links, rechts)
            if ergebnis(a, b) == 1:
                siege = siege + 1
        siegprozent[(links, rechts)] = round(100 * siege / anzahl)

print(' ' * 12, ' '.join(n[:4].rjust(4) for n in namen), ' Summe')
for links in namen:
    zeile = [siegprozent[(links, rechts)] for rechts in namen]
    print(links.ljust(12), ' '.join(str(x).rjust(4) for x in zeile), str(sum(zeile)).rjust(6))

             alph kons sech line line mehr mehr zufa alte kopi letz tren  Summe
alphasoft       0    0    0    0    0    0    0    0    0    0    0    0      0
konstant101   100    0    0    0    0    0    0    0    0    0    0    0    100
sechs166      100  100    0  100    0    0    0   97  100  100  100  100    797
linear_auf    100  100    0    0    0  100    0    4    0  100  100    0    504
linear_ab     100  100  100    0    0    0    0   94  100    0    0    0    494
mehr1         100  100  100    0  100    0    0   54    0    0    0  100    554
mehr20        100  100  100  100  100  100    0   72  100    0    0  100    872
zufall        100   98    0   95    4   18   10   42   46   30   96   94    633
alternative   100  100    0  100    0  100    0   53    0    0    0  100    553
kopie           0    0    0    0  100  100  100   46  100    0    0    0    446
letztes       100  100    0    0  100    0  100    6  100  100    0    0    606
trend         100  100    0  100  100   

### Was die Tabelle zeigt

- **Alphasoft verliert gegen fast alle.** Die Spalte `alphasoft` ist bei allen Spielern 100, außer bei `alphasoft` selbst und bei `kopie`. Beide bieten in jeder Runde 100, es steht unentschieden, auch beim Gold: Das zählt nicht als Sieg. Auch `zufall` (100 bis 199) schlägt Alphasoft in allen 200 Spielen.
- **Kein Spieler schlägt alle.** Die Beziehung „schlägt“ ist keine Rangordnung, sie kann im Kreis laufen: `sechs166` schlägt `linear_auf`, `linear_auf` schlägt `mehr1`, und `mehr1` schlägt `sechs166` (alle drei Felder stehen mit 100 in der Tabelle). Es gibt also **kein bestes Programm**; wie ein Programm abschneidet, hängt davon ab, wer sonst im Turnier mitspielt. Das sagen auch die Lösungshinweise: Die wirkliche Güte zeigt sich erst im Turnier.
- **Die Summenspalte ist mit Vorsicht zu lesen.** Sie hängt von der Wahl unserer Gegner ab. Vorn liegt `mehr20` mit 872, danach `sechs166` mit 797. Unsere beiden adaptiven Spieler `letztes` und `trend` kommen auf 606, die „Alternative“ der Hinweise auf 553. Die adaptiven Spieler sind hier also **nicht** besser als feste Folgen; sie sind sehr einfach gebaut (sie kennen nur das letzte Gebot oder den letzten Anstieg). Die Hinweise halten adaptive Programme für stärker, wenn sie das Muster des Gegners wirklich erkennen; das ist deutlich mehr Arbeit.

## Die Strategie, so wie man sie in der Dokumentation erläutert

Das Verfahren, das **Alphasoft garantiert** schlägt: *Biete in mindestens 6 der 10 Runden mehr als 100 Unzen (insgesamt höchstens 1000).* Dann gewinnt man mindestens 6 Runden, also mindestens 12 von 20 Gramm, Alphasoft höchstens 8. Das Gold spielt keine Rolle mehr. (Die Probe mit 5000 Folgen oben bestätigt es; der Beweis ist die Rechnung 6 · 2 = 12 > 8.)

Für **unbekannte Gegner** gelten die Überlegungen der Hinweise („Die Alternative“), die man in die Dokumentation schreibt:

1. Mindestens 6 Gebote über 100, damit Alphasoft sicher verliert.
2. Nicht alles Gold ausgeben: Gewinnt jeder 5 Runden, entscheidet das Gold. Die Folge `10, 165, 101, 2, 166, 4, 14, 170, 142, 165` kostet 939 Unzen, 61 bleiben übrig.
3. Hohe und niedrige Gebote im Wechsel, damit „Immer-eins-mehr“- und linear steigende oder fallende Gegner nicht jede zweite Runde gewinnen.
4. Kein Gebot 0, vor allem nicht am Ende: Manche Gegner haben sonst früh kein Gold mehr.
5. Adaptive Gegner kann man nicht gezielt schlagen; für sie wirkt eine solche Folge wie ein Zufallsspieler.

Ein **adaptives Programm** wie `letztes` beobachtet den Gegner und bietet knapp darüber, solange es für die noch fehlenden Siege reicht. Es schlägt jeden konstant bietenden Gegner; ein Gegner wie `mehr1` bringt es dagegen höchstens zum Unentschieden. Man schreibt Verfahren und Begründung auf, wie es die Bewertungskriterien der Hinweise verlangen.

## Zum Schluss: Eine feste Folge mit MiniZinc suchen

Statt die Folge von Hand zu bauen, kann man sie **suchen lassen**: Wir geben eine Liste von Gegnerfolgen vor und fragen, welche eigene Folge (10 Gebote, zusammen höchstens 1000 Unzen) gegen möglichst viele davon gewinnt. Das passt zu MiniZinc: eine Liste von Variablen `x`, eine Zählung der Siege, `maximize`. Die Siegbedingung ist die Regel der Aufgabe: Gramm zählen (`2 * (x[r] > geb[g, r]) + (x[r] == geb[g, r])`), bei genau 10 Gramm entscheidet das Gold (`sum(x) < Summe der Gegnergebote`).

**Gegnerfolgen** (alle feste Zahlenfolgen): Alphasoft; konstante Folgen 50, 60, 70, 80, 90; 9-mal 101 und dann 91; sechsmal 101, 120, 142 oder 166 und danach viermal 1; linear steigende und fallende Folgen mit Schrittweite 5, 10, 20 und 22; die „Alternative“ der Hinweise. Reaktive Gegner wie `mehr1` fehlen, denn deren Gebote hängen von unseren ab; das ließe sich mit mehr Aufwand auch in MiniZinc ausdrücken.

In [ ]:
gegner_folgen = {'alphasoft': [100] * 10}
for k in range(50, 100, 10):
    gegner_folgen['konstant' + str(k)] = [k] * 10
gegner_folgen['konstant101'] = [101] * 9 + [91]
for k in [101, 120, 142, 166]:
    gegner_folgen['sechs' + str(k)] = [k] * 6 + [1] * 4
for c in [5, 10, 20, 22]:
    a = (1000 - 45 * c) // 10
    gegner_folgen['auf' + str(c)] = [a + c * i for i in range(10)]
    gegner_folgen['ab' + str(c)] = [a + c * (9 - i) for i in range(10)]
gegner_folgen['alternative'] = [10, 165, 101, 2, 166, 4, 14, 170, 142, 165]


def als_dzn(folgen):
    """Datendatei: Anzahl der Gegner und ihre Gebote als Tabelle (eine Zeile je Gegner)."""
    zahlen = [str(g) for f in folgen for g in f]
    n = str(len(folgen))
    return 'G = ' + n + ';\ngeb = array2d(1..' + n + ', 1..10, [' + ', '.join(zahlen) + ']);'


for name in gegner_folgen:
    assert sum(gegner_folgen[name]) <= 1000
print(len(gegner_folgen), 'Gegnerfolgen')
print(als_dzn(list(gegner_folgen.values()))[:100], '...')

20 Gegnerfolgen
G = 20;
geb = array2d(1..20, 1..10, [100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 50, 50, 50, 5 ...


Die Modelldatei (`G` und `geb` kommen aus der Datendatei, die eine Python-Zelle `als_dzn` aus den Folgen erzeugt, siehe oben):

```minizinc
int: G;                            % Anzahl der Gegnerfolgen
set of int: GEGNER = 1..G;
set of int: RUNDE = 1..10;
array[GEGNER, RUNDE] of int: geb;  % geb[g, r]: Gebot des Gegners g in Runde r

array[RUNDE] of var 0..1000: x;    % unsere Gebote
constraint sum(x) <= 1000;

% Gramm, die wir gegen Gegner g bekommen
array[GEGNER] of var int: gramm = [sum(r in RUNDE)(2 * bool2int(x[r] > geb[g, r]) + bool2int(x[r] == geb[g, r]))
                                   | g in GEGNER];
% Sieg: mehr als 10 Gramm, oder genau 10 und weniger Gold ausgegeben
array[GEGNER] of var bool: sieg = [gramm[g] > 10 \/ (gramm[g] == 10 /\ sum(x) < sum(r in RUNDE)(geb[g, r]))
                                   | g in GEGNER];

solve maximize sum(g in GEGNER)(bool2int(sieg[g]));
output ["Siege: ", show(sum(g in GEGNER)(bool2int(sieg[g]))), " von ", show(G), "\n", "x = ", show(x), "\n"];
```

**Ausgabe** (Chuffed):

```
Siege: 20 von 20
x = [167, 3, 3, 167, 110, 113, 133, 52, 189, 1]
```

- `var 0..1000: x`: unsere zehn Gebote sind die Unbekannten. `sum(x) <= 1000` ist die Goldgrenze.
- `gramm[g]` zählt für jede Gegnerfolge die Gramm. `bool2int` macht aus einem Vergleich eine 0 oder 1.
- `sieg[g]` fasst die Siegbedingung der Aufgabe in eine Bedingung (`\/` heißt „oder“, `/\` heißt „und“).
- `solve maximize`: Die Zahl der besiegten Gegner soll möglichst groß sein.

**Messung** (alle 20 Gegnerfolgen, das ist die einzige Datenmenge dieser Aufgabe): **Chuffed** und **CP-SAT** finden eine Folge mit 20 von 20 Siegen und beweisen, dass mehr nicht geht, beide in wenigen Sekunden. **Gecode** ist nach einer Minute nicht fertig und hat bis dahin nur eine Folge mit 2 von 20 Siegen gefunden. Die Wahl des Solvers entscheidet hier also, ob das Modell überhaupt brauchbar ist.

Die Python-Zelle unten prüft die gefundene Folge mit unserer Funktion `ergebnis` nach und lässt sie dann gegen die **anderen** Spieler aus dem Turnier antreten, die MiniZinc nicht kannte (reaktive und zufällige Gegner):

In [ ]:
FOLGE_MZN = [167, 3, 3, 167, 110, 113, 133, 52, 189, 1]
print('Folge:', FOLGE_MZN, 'Summe', sum(FOLGE_MZN))
siege_von_20 = sum(1 for f in gegner_folgen.values() if ergebnis(FOLGE_MZN, f) == 1)
print('Besiegte Gegnerfolgen:', siege_von_20, 'von', len(gegner_folgen))


def spiele_folge(folge, gegner):
    """Wie spiele(), aber unser Spieler bietet eine feste Folge (nie mehr als sein Gold)."""
    meine = []
    ihre = []
    rest_m = 1000
    rest_i = 1000
    for runde in range(1, 11):
        m = min(folge[runde - 1], rest_m)
        i = gebot(gegner, runde, ihre, meine, rest_i)
        meine.append(m)
        ihre.append(i)
        rest_m = rest_m - m
        rest_i = rest_i - i
    return meine, ihre


random.seed(2013)
for gegner in ['mehr1', 'mehr20', 'kopie', 'zufall', 'letztes', 'trend']:
    for name, folge in [('MiniZinc-Folge', FOLGE_MZN), ('Alternative  ', FOLGE_ALTERNATIVE)]:
        anzahl = 200 if gegner == 'zufall' else 1
        siege = 0
        for _ in range(anzahl):
            a, b = spiele_folge(folge, gegner)
            if ergebnis(a, b) == 1:
                siege = siege + 1
        print(name, 'gegen', gegner.ljust(7), round(100 * siege / anzahl), '%')

Folge: [167, 3, 3, 167, 110, 113, 133, 52, 189, 1] Summe 938
Besiegte Gegnerfolgen: 20 von 20
MiniZinc-Folge gegen mehr1   100 %
Alternative   gegen mehr1   100 %
MiniZinc-Folge gegen mehr20  0 %
Alternative   gegen mehr20  0 %
MiniZinc-Folge gegen kopie   100 %
Alternative   gegen kopie   0 %
MiniZinc-Folge gegen zufall  91 %
Alternative   gegen zufall  47 %
MiniZinc-Folge gegen letztes 100 %
Alternative   gegen letztes 0 %
MiniZinc-Folge gegen trend   100 %
Alternative   gegen trend   100 %


**Was man daraus lernt:**

- Die gefundene Folge `167, 3, 3, 167, 110, 113, 133, 52, 189, 1` (938 Unzen, kein Gebot 0, hohe und niedrige Gebote gemischt) schlägt alle 20 Gegnerfolgen, für die sie gesucht wurde. Sie verhält sich also ähnlich wie die „Alternative“ der Hinweise, nur dass der Solver sie für uns gefunden hat.
- Gegen die Spieler, die MiniZinc **nicht** kannte, schneidet sie besser ab als die „Alternative“: gegen `kopie` und `letztes` gewinnt sie jedes Spiel, die „Alternative“ keines, gegen `zufall` 91 % statt 47 %. Gegen `mehr20` verlieren beide. Ein Grund: Die Folge wurde nur gegen die Gegner optimiert, die wir aufgelistet haben. Wer die Folge kennt, schlägt sie (siehe unten).
- Das bestätigt die Aussage der Hinweise: Es gibt keine beste Folge, nur eine, die gegen die erwarteten Gegner gut ist. Je besser man die Gegner kennt (oder rät), desto besser kann man sie schlagen, und ein Solver nimmt einem die Suche ab.

## Für Neugierige: Warum es kein bestes Programm gibt

Angenommen, jemand kennt die feste Gebotsfolge `x` seines Gegners (zusammen höchstens 1000 Unzen). Dann gewinnt er immer: Er sucht die sechs **kleinsten** Gebote von `x`; sie kosten zusammen höchstens 6/10 von 1000, also höchstens 600 Unzen. In genau diesen sechs Runden bietet er 1 mehr als `x`, in den anderen vier 0. Das kostet höchstens 606 Unzen, bringt 12 Gramm und schlägt `x`. Jede feste Folge hat also eine Folge, die sie schlägt (die Zelle unten probiert es an 1000 Zufallsfolgen aus). Das Muster ist wie bei „Schere, Stein, Papier“: Jede feste Strategie hat eine passende Antwort, und die Antwort hat wieder eine. Das Dreieck `sechs166` → `linear_auf` → `mehr1` → `sechs166` aus der Tabelle ist ein Beispiel.

In [ ]:
def gegenfolge(x):
    """Eine Folge, die x schlägt: 1 mehr als x in den sechs Runden mit den kleinsten Geboten, sonst 0."""
    paare = sorted([(x[r], r) for r in range(10)])     # (Gebot, Runde), nach dem Gebot sortiert
    kleinste = [r for gebot_r, r in paare[:6]]         # die Runden der sechs kleinsten Gebote
    return [x[r] + 1 if r in kleinste else 0 for r in range(10)]


random.seed(7)
folgen = [FOLGE_MZN, FOLGE_ALTERNATIVE]
while len(folgen) < 1000:
    f = [random.randint(0, 300) for _ in range(10)]
    if sum(f) <= 1000:
        folgen.append(f)
for f in folgen:
    assert sum(gegenfolge(f)) <= 606 and ergebnis(gegenfolge(f), f) == 1
print(len(folgen), 'Folgen, zu jeder gibt es eine Gegenfolge, die gewinnt, z. B. gegen die Alternative:')
print(gegenfolge(FOLGE_ALTERNATIVE))

1000 Folgen, zu jeder gibt es eine Gegenfolge, die gewinnt, z. B. gegen die Alternative:
[11, 0, 102, 3, 0, 5, 15, 0, 143, 0]


Gewinnen kann man in diesem Spiel nur durch **Wissen über den Gegner**. Deshalb ist die Aufgabe so gestellt, dass Alphasoft bekannt ist (garantiert schlagbar), und für unbekannte Gegner bleibt nur die Mischung aus Beobachten (adaptiv) und einer Folge, die gegen viele typische Programme gut ist.